# 🤖 Lab 6: AI-Assisted Kubernetes Cluster Resources with `kubectl-ai`
### AIOps Training — Session 6

---

## 🎯 Lab Objectives

By the end of this lab, you will be able to:

1. Install and configure **Ollama** with a local LLM (`llama3.2`)
2. Install and configure **`kubectl-ai`** to use a locally hosted model
3. Use natural language queries to interact with a **Kubernetes cluster**
4. Perform AI-assisted diagnostics on **Pods, Deployments, Secrets, and Services**
5. Leverage AI to troubleshoot common Kubernetes issues intelligently

---

## 🏗️ Lab Environment

| Component | Details |
|---|---|
| **Kubernetes Options** | k3s (local) **or** OCI Kubernetes Cluster |
| **LLM Provider** | Ollama (local) |
| **Model** | `llama3.2` |
| **Interface** | `kubectl-ai` CLI plugin |
| **Lab Interface** | JupyterLab |

> **⚠️ Note:** All shell commands in this notebook use the `!` prefix (single command) or `%%bash` (multi-line). Choose your cluster context before starting — **k3s** or **OCI**. Steps are compatible with both.

---

## 📦 Prerequisites Check

In [ ]:
# Verify baseline tools are available
import subprocess, sys

tools = {
    "kubectl": "kubectl version --client --short 2>/dev/null || kubectl version --client",
    "curl": "curl --version | head -1",
    "git": "git --version",
}

print('🔍 Checking prerequisites...\n')
for tool, cmd in tools.items():
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    status = '✅' if result.returncode == 0 else '❌ NOT FOUND'
    output = result.stdout.strip() or result.stderr.strip()
    print(f'{status} {tool}: {output[:80]}')

# Check Go separately — it will be installed in Part 3 if missing
go_result = subprocess.run('go version', shell=True, capture_output=True, text=True)
go_status = '✅' if go_result.returncode == 0 else '⚠️  NOT FOUND (will be installed in Part 3)'
go_output = go_result.stdout.strip() or ''
print(f'{go_status} go: {go_output}')


In [ ]:
# Verify Kubernetes cluster connectivity
print("🔍 Checking Kubernetes cluster...\n")
!kubectl cluster-info
print("\n📋 Current context:")
!kubectl config current-context
print("\n🖥️  Available contexts:")
!kubectl config get-contexts

---

## 🦙 Part 1: Install Ollama

**Ollama** is a tool for running large language models locally. It provides a simple REST API that `kubectl-ai` will use as its AI backend.

### Architecture

```
 ┌───────────────────┐        ┌──────────────────────┐        ┌─────────────────────┐
 │   kubectl-ai      │──────▶│   Ollama API          │──────▶│   llama3.2 Model    │
 │   (CLI Plugin)    │       │   localhost:11434     │       │   (local weights)   │
 └───────────────────┘        └──────────────────────┘        └─────────────────────┘
         │
         ▼
 ┌───────────────────┐
 │  Kubernetes API   │
 │  (k3s / OCI)      │
 └───────────────────┘
```

### Step 1.1 — Install Ollama

In [ ]:
%%bash
# Check if Ollama is already installed
if command -v ollama &> /dev/null; then
    echo "✅ Ollama already installed: $(ollama --version)"
else
    echo "📥 Installing Ollama..."
    curl -fsSL https://ollama.com/install.sh | sh
    echo "✅ Ollama installed successfully!"
fi

### Step 1.2 — Start Ollama Service

> **Note:** Ollama needs to run as a background daemon. In JupyterLab, we start it in the background using `nohup`. If Ollama was installed via a system package or snap, it may already be running.

In [ ]:
%%bash
# Check if Ollama service is already running
if curl -s http://localhost:11434 &> /dev/null; then
    echo "✅ Ollama server is already running on port 11434"
else
    echo "🚀 Starting Ollama server in background..."
    nohup ollama serve > /tmp/ollama.log 2>&1 &
    sleep 3
    if curl -s http://localhost:11434 &> /dev/null; then
        echo "✅ Ollama server started successfully!"
    else
        echo "⏳ Waiting for server to be ready..."
        sleep 5
        curl -s http://localhost:11434 && echo "✅ Ready!" || echo "❌ Server failed to start. Check /tmp/ollama.log"
    fi
fi

In [ ]:
%%bash
# Verify Ollama API is responding
echo "🔗 Testing Ollama API endpoint..."
curl -s http://localhost:11434/api/tags | python3 -m json.tool 2>/dev/null || echo "Ollama is running (no models pulled yet)"
echo ""
echo "📋 Ollama version:"
ollama --version

---

## 🧠 Part 2: Pull the `llama3.2` Model

**llama3.2** is Meta's compact, high-performance language model. It runs efficiently on local hardware and is well-suited for Kubernetes operations tasks.

| Model | Parameters | Size | Use Case |
|---|---|---|---|
| `llama3.2` | 3B | ~2 GB | Fast, good for ops tasks |
| `llama3.2:1b` | 1B | ~1.3 GB | Minimal resources |
| `llama3.2:3b` | 3B | ~2 GB | Default (recommended) |

### Step 2.1 — Pull llama3.2

In [ ]:
%%bash
echo "📥 Pulling llama3.2 model (this may take several minutes on first run)..."
echo "Model size: ~2 GB — please wait"
echo "─────────────────────────────────────────"
ollama pull llama3.2
echo "─────────────────────────────────────────"
echo "✅ Model pulled successfully!"

### Step 2.2 — Verify Model & Run a Quick Test

In [ ]:
%%bash
echo "📋 Available Ollama models:"
ollama list

In [ ]:
%%bash
# Quick smoke test via Ollama REST API
echo "🧪 Testing llama3.2 via Ollama API..."
echo ""

curl -s http://localhost:11434/api/generate \
  -H 'Content-Type: application/json' \
  -d '{
    "model": "llama3.2",
    "prompt": "In one sentence, what is Kubernetes?",
    "stream": false
  }' | python3 -c "
import json, sys
data = json.load(sys.stdin)
print('🤖 Model response:')
print(data.get('response', 'No response received'))
print(f\"\\n⚡ Done in {data.get('total_duration', 0) / 1e9:.2f} seconds\")
"

---

## 🔌 Part 3: Install `kubectl-ai`

**`kubectl-ai`** is a `kubectl` plugin that translates natural language into Kubernetes operations. It uses an LLM backend (Ollama in our case) to understand your intent and generate the correct `kubectl` commands or YAML manifests.

### How it works:

```
You: "Show me all pods that are not Running"
        │
        ▼
kubectl-ai  ──── (sends to llama3.2 via Ollama) ────▶  kubectl get pods --all-namespaces | grep -v Running
        │
        ▼
Executes against your cluster and returns results
```

Since Go is **not pre-installed** on this system, we will:
1. Install Go (required for some build tools)
2. Install `kubectl-ai` via the **official install script** (recommended)
3. Verify the installation

> **💡 Note:** The `go install` path changed in v0.0.31. The official `install.sh` script
> is now the recommended method — it auto-detects your platform and downloads the right binary.
> Step 3.3 provides a manual binary download as a fallback.

---

### Step 3.1 — Install Go

In [ ]:
%%bash
# Install Go (latest stable) if not already present
if command -v go &> /dev/null; then
    echo "✅ Go already installed: $(go version)"
    exit 0
fi

echo "📥 Fetching latest stable Go version..."
GO_VERSION=$(curl -s 'https://go.dev/dl/?mode=json' | \
    python3 -c "import json,sys; releases=json.load(sys.stdin); print(releases[0]['version'])")

# Fallback if the above fails
GO_VERSION=${GO_VERSION:-go1.23.5}
echo "📌 Installing $GO_VERSION ..."

# Detect architecture
ARCH=$(uname -m)
case $ARCH in
    x86_64)  ARCH="amd64" ;;
    aarch64) ARCH="arm64" ;;
    armv7l)  ARCH="armv6l" ;;
esac

TARBALL="${GO_VERSION}.linux-${ARCH}.tar.gz"
DOWNLOAD_URL="https://dl.google.com/go/${TARBALL}"
echo "⬇️  Downloading: $DOWNLOAD_URL"

curl -L "$DOWNLOAD_URL" -o /tmp/go.tar.gz

# Install to /usr/local/go (standard location)
sudo rm -rf /usr/local/go
sudo tar -C /usr/local -xzf /tmp/go.tar.gz
rm /tmp/go.tar.gz

# Make go available in current session
export PATH=$PATH:/usr/local/go/bin

# Persist to shell profiles
for PROFILE in ~/.bashrc ~/.bash_profile ~/.profile; do
    grep -qxF 'export PATH=$PATH:/usr/local/go/bin' "$PROFILE" 2>/dev/null || \
        echo 'export PATH=$PATH:/usr/local/go/bin' >> "$PROFILE"
done

echo ""
echo "✅ Go installed successfully!"
go version

In [ ]:
%%bash
# Source PATH updates and confirm Go is available
export PATH=$PATH:/usr/local/go/bin

echo "🔍 Go binary location: $(which go)"
echo "📦 Go version:         $(go version)"
echo "📁 GOPATH:             $(go env GOPATH)"
echo "📁 GOROOT:             $(go env GOROOT)"

### Step 3.2 — Install `kubectl-ai` via Official Install Script *(Recommended)*


In [ ]:
%%bash
export PATH=$PATH:/usr/local/go/bin

# The official install script handles platform detection and binary download
# This replaces the old 'go install' method — the module path changed in v0.0.31+
echo "📥 Installing kubectl-ai via official install script..."
curl -sSL https://raw.githubusercontent.com/GoogleCloudPlatform/kubectl-ai/main/install.sh | bash

# The script installs to ~/.local/bin by default (no sudo needed)
export PATH=$PATH:$HOME/.local/bin

# Persist to shell profiles
for PROFILE in ~/.bashrc ~/.bash_profile ~/.profile; do
    grep -qxF 'export PATH=$PATH:$HOME/.local/bin' "$PROFILE" 2>/dev/null || \
        echo 'export PATH=$PATH:$HOME/.local/bin' >> "$PROFILE"
done

echo ""
if command -v kubectl-ai &>/dev/null; then
    echo "✅ kubectl-ai installed successfully!"
    kubectl-ai --help 2>&1 | head -5
else
    echo "⚠️  Not in PATH yet. Run Step 3.3 (binary fallback) below."
fi


### Step 3.3 — Alternative: Install via Pre-built Binary *(skip if Step 3.2 succeeded)*

> Use this **only** if `go install` above failed. This downloads a pre-compiled binary directly from GitHub Releases — no Go required.

In [ ]:
%%bash
# Skip if kubectl-ai already installed
export PATH=$PATH:/usr/local/go/bin:$HOME/.local/bin:/usr/local/bin
if command -v kubectl-ai &>/dev/null; then
    echo "✅ kubectl-ai already installed — skipping binary fallback."
    exit 0
fi

echo "📥 Fallback: downloading pre-built binary directly from GitHub Releases..."

OS=$(uname -s | tr '[:upper:]' '[:lower:]')
ARCH=$(uname -m)
case $ARCH in
    x86_64)  ARCH="x86_64" ;;
    aarch64) ARCH="arm64"  ;;
    armv7l)  ARCH="arm"    ;;
esac

# Capitalise OS for asset name (Linux, Darwin)
OS_CAP=$(uname -s)  # e.g. Linux

# Use GitHub redirect for /latest/download — avoids API rate limits
BINARY_URL="https://github.com/GoogleCloudPlatform/kubectl-ai/releases/latest/download/kubectl-ai_${OS_CAP}_${ARCH}.tar.gz"
echo "⬇️  $BINARY_URL"

curl -L --fail --show-error "$BINARY_URL" -o /tmp/kubectl-ai.tar.gz
if [ $? -ne 0 ]; then
    echo "❌ Download failed. Check URL or network and retry."
    exit 1
fi

tar -xzf /tmp/kubectl-ai.tar.gz -C /tmp/
mkdir -p ~/.local/bin
mv /tmp/kubectl-ai ~/.local/bin/kubectl-ai
chmod +x ~/.local/bin/kubectl-ai

export PATH=$PATH:$HOME/.local/bin

echo ""
kubectl-ai --help 2>&1 | head -5 && echo "✅ kubectl-ai binary installed!"


In [ ]:
%%bash
export PATH=$PATH:/usr/local/go/bin:$(go env GOPATH 2>/dev/null)/bin:$HOME/.local/bin:/usr/local/bin

echo "🔍 Locating kubectl-ai binary:"
which kubectl-ai 2>/dev/null || \
    find $HOME/.local/bin /usr/local/bin -name 'kubectl-ai' 2>/dev/null | head -3

echo ""
echo "📋 kubectl-ai help (first 15 lines):"
kubectl-ai --help 2>&1 | head -15


---

## ⚙️ Part 4: Configure `kubectl-ai` with Local Ollama Model

By default, `kubectl-ai` expects an OpenAI-compatible API key. We need to configure it to point to our **local Ollama instance** instead.

`kubectl-ai` supports Ollama via the `--llm-provider` and `--model` flags, or a persistent configuration file.

### Step 4.1 — Create Configuration File

In [ ]:
%%bash
mkdir -p ~/.config/kubectl-ai

cat > ~/.config/kubectl-ai/config.yaml << 'EOF'
# kubectl-ai Configuration — Ollama local LLM
llmProvider: ollama
model: llama3.2

# llama3.2 requires the tool-use shim for proper function calling
enableToolUseShim: true

# Safety: require confirmation before resource-modifying commands
skipPermissions: false
EOF

echo "✅ Config written to ~/.config/kubectl-ai/config.yaml"
echo ""
cat ~/.config/kubectl-ai/config.yaml


### Step 4.2 — Configure via Environment Variables

You can also configure kubectl-ai using environment variables (useful for scripting):

In [ ]:
import os

# Set environment variables for kubectl-ai
os.environ['KUBECTL_AI_LLM_PROVIDER'] = 'ollama'
os.environ['KUBECTL_AI_MODEL'] = 'llama3.2'
os.environ['OLLAMA_HOST'] = 'http://localhost:11434'

# Extend PATH to include Go and GOPATH bin directories
import subprocess
go_bin = subprocess.run('go env GOPATH 2>/dev/null', shell=True,
                        capture_output=True, text=True,
                        env={**os.environ, 'PATH': os.environ['PATH'] + ':/usr/local/go/bin'}).stdout.strip()
extra_paths = '/usr/local/go/bin:/usr/local/bin'
if go_bin:
    extra_paths = f'{go_bin}/bin:{extra_paths}'
os.environ['PATH'] = f"{os.environ['PATH']}:{extra_paths}"

print('✅ Environment variables configured:')
for k in ['KUBECTL_AI_LLM_PROVIDER', 'KUBECTL_AI_MODEL', 'OLLAMA_HOST']:
    print(f'   {k} = {os.environ.get(k)}')
os.environ['KUBECTL_AI_ENABLE_TOOL_USE_SHIM'] = 'true'
print('✅ Environment variables configured:')
for k in ['KUBECTL_AI_LLM_PROVIDER', 'KUBECTL_AI_MODEL', 'OLLAMA_HOST', 'KUBECTL_AI_ENABLE_TOOL_USE_SHIM']:
    print(f'   {k} = {os.environ.get(k)}')
print(f'   PATH includes: ~/.local/bin, /usr/local/go/bin')


### Step 4.3 — Test kubectl-ai with Ollama

In [ ]:
%%bash
export PATH=$PATH:$(go env GOPATH 2>/dev/null)/bin:/usr/local/bin
export KUBECTL_AI_LLM_PROVIDER=ollama
export KUBECTL_AI_MODEL=llama3.2
export OLLAMA_HOST=http://localhost:11434

echo "🧪 Test 1: Simple cluster query via kubectl-ai"
echo "──────────────────────────────────────────────"
kubectl-ai --llm-provider ollama --model llama3.2 --enable-tool-use-shim \
  "List all namespaces in the cluster" 2>&1 || \
kubectl-ai "List all namespaces in the cluster" 2>&1

In [ ]:
%%bash
export PATH=$PATH:$(go env GOPATH 2>/dev/null)/bin:/usr/local/bin
export KUBECTL_AI_LLM_PROVIDER=ollama
export KUBECTL_AI_MODEL=llama3.2
export OLLAMA_HOST=http://localhost:11434

echo "🧪 Test 2: Node health query"
echo "──────────────────────────────────────────────"
kubectl-ai --llm-provider ollama --model llama3.2 --enable-tool-use-shim \
  "Show me all nodes and their status" 2>&1

### 💡 Helper Function for Lab

For convenience, let's create a Python helper that wraps `kubectl-ai` calls with proper env vars:

In [ ]:
import subprocess, os, sys

def kai(query, dry_run=False, namespace=None):
    """
    kubectl-ai helper function.

    Args:
        query    : Natural language query string
        dry_run  : If True, only show generated command (don't execute)
        namespace: Optional Kubernetes namespace context
    """
    env = os.environ.copy()
    # Build PATH with all Go-related locations
    go_path = subprocess.run(
        'go env GOPATH 2>/dev/null',
        shell=True, capture_output=True, text=True,
        env={**env, 'PATH': env.get('PATH','') + ':/usr/local/go/bin'}
    ).stdout.strip()
    home = os.path.expanduser('~')
    env['PATH'] = (
        f"{env.get('PATH', '')}:{home}/.local/bin:/usr/local/go/bin:/usr/local/bin"
        + (f":{go_path}/bin" if go_path else "")
    )
    env['KUBECTL_AI_LLM_PROVIDER'] = 'ollama'
    env['KUBECTL_AI_MODEL'] = 'llama3.2'
    env['OLLAMA_HOST'] = 'http://localhost:11434'

    cmd = ['kubectl-ai', '--llm-provider', 'ollama', '--model', 'llama3.2', '--enable-tool-use-shim']
    if dry_run:
        cmd.append('--dry-run')
    if namespace:
        cmd.extend(['-n', namespace])
    cmd.append(query)

    print(f'🤖 Query: {query}')
    if namespace:
        print(f'📁 Namespace: {namespace}')
    print('─' * 60)

    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    output = result.stdout or result.stderr
    print(output)
    return output

print('✅ kai() helper function registered. Usage: kai("your natural language query")')


---

## 🚀 Part 5: AI-Assisted Kubernetes Operations

Now let's use `kubectl-ai` with natural language queries to manage and diagnose Kubernetes resources.

### 5.1 — Setup: Create Demo Namespace & Resources

We'll create sample resources in a `lab6-demo` namespace to practice with.

In [ ]:
%%bash
# Create demo namespace
kubectl create namespace lab6-demo --dry-run=client -o yaml | kubectl apply -f -
echo "✅ Namespace lab6-demo ready"

In [ ]:
%%bash
# Deploy sample applications for the lab
cat <<'EOF' | kubectl apply -f -
---
# Deployment 1: nginx web server (healthy)
apiVersion: apps/v1
kind: Deployment
metadata:
  name: nginx-demo
  namespace: lab6-demo
  labels:
    app: nginx-demo
    tier: frontend
spec:
  replicas: 3
  selector:
    matchLabels:
      app: nginx-demo
  template:
    metadata:
      labels:
        app: nginx-demo
        tier: frontend
    spec:
      containers:
      - name: nginx
        image: nginx:1.25
        ports:
        - containerPort: 80
        resources:
          requests:
            memory: "64Mi"
            cpu: "100m"
          limits:
            memory: "128Mi"
            cpu: "200m"
---
# Deployment 2: Intentionally broken (bad image)
apiVersion: apps/v1
kind: Deployment
metadata:
  name: broken-app
  namespace: lab6-demo
  labels:
    app: broken-app
spec:
  replicas: 2
  selector:
    matchLabels:
      app: broken-app
  template:
    metadata:
      labels:
        app: broken-app
    spec:
      containers:
      - name: broken
        image: nginx:this-tag-does-not-exist
        ports:
        - containerPort: 8080
---
# Service for nginx-demo
apiVersion: v1
kind: Service
metadata:
  name: nginx-demo-svc
  namespace: lab6-demo
spec:
  selector:
    app: nginx-demo
  ports:
  - port: 80
    targetPort: 80
  type: ClusterIP
---
# Secret: sample app credentials
apiVersion: v1
kind: Secret
metadata:
  name: app-credentials
  namespace: lab6-demo
type: Opaque
stringData:
  username: admin
  password: SuperSecret123!
  db-connection: postgresql://localhost:5432/myapp
---
# ConfigMap
apiVersion: v1
kind: ConfigMap
metadata:
  name: app-config
  namespace: lab6-demo
data:
  LOG_LEVEL: "info"
  MAX_CONNECTIONS: "100"
  ENVIRONMENT: "training-lab"
EOF

echo ""
echo "⏳ Waiting for pods to initialize..."
sleep 8
echo ""
echo "📋 Resources in lab6-demo namespace:"
kubectl get all -n lab6-demo

---

### 5.2 — Pod Management with kubectl-ai

Let's use natural language to inspect and troubleshoot pods.

In [ ]:
# Query 1: List all pods with their status
kai("List all pods in the lab6-demo namespace and show their status and which node they are running on")

In [ ]:
# Query 2: Find failing pods
kai("Find all pods that are not in Running state across all namespaces and explain what might be wrong")

In [ ]:
# Query 3: Troubleshoot the broken pod
kai("Describe the broken-app pods in lab6-demo namespace and tell me why they are failing")

In [ ]:
# Query 4: Get logs from nginx pods
kai("Show me the last 20 log lines from the nginx-demo pods in lab6-demo")

In [ ]:
# Query 5: Resource usage
kai("Show resource requests and limits for all pods in lab6-demo namespace")

---

### 5.3 — Deployment Operations with kubectl-ai

In [ ]:
# Query 6: List all deployments with readiness
kai("List all deployments in lab6-demo and show how many replicas are ready vs desired")

In [ ]:
# Query 7: Deployment rollout status
kai("Check the rollout status of the nginx-demo deployment in lab6-demo namespace")

In [ ]:
# Query 8: Scale a deployment
kai("Scale the nginx-demo deployment in lab6-demo namespace to 5 replicas", dry_run=True)

In [ ]:
# Query 9: Rollout history
kai("Show the rollout history of nginx-demo deployment in lab6-demo")

In [ ]:
# Query 10: Update image (dry run for safety)
kai("Update the nginx-demo deployment in lab6-demo to use nginx:1.26 image", dry_run=True)

---

### 5.4 — Secrets & ConfigMaps with kubectl-ai

> **🔒 Security Note:** kubectl-ai intelligently avoids revealing secret values directly. It will help you manage secret metadata and keys without exposing sensitive data.

In [ ]:
# Query 11: List secrets
kai("List all secrets in the lab6-demo namespace and show their types and keys (not values)")

In [ ]:
# Query 12: Inspect a specific secret's structure
kai("Describe the app-credentials secret in lab6-demo and list what keys it contains")

In [ ]:
# Query 13: Create a new secret
kai("""
Create a new secret named 'db-secret' in lab6-demo namespace 
with keys: DB_HOST=postgres-svc, DB_PORT=5432, DB_NAME=appdb
""", dry_run=True)

In [ ]:
# Query 14: ConfigMap operations
kai("Show me all configmaps in lab6-demo namespace and their data")

In [ ]:
# Query 15: Check which pods use which secrets
kai("Which pods in lab6-demo namespace are using secrets or configmaps as environment variables or volume mounts?")

---

### 5.5 — Service Operations with kubectl-ai

In [ ]:
# Query 16: List services
kai("List all services in lab6-demo namespace and show their type, cluster IP, and port mappings")

In [ ]:
# Query 17: Check service endpoints
kai("Check the endpoints for nginx-demo-svc in lab6-demo and verify it has healthy pod endpoints")

In [ ]:
# Query 18: Expose a deployment as a service (dry-run)
kai("Create a NodePort service for the nginx-demo deployment in lab6-demo on port 8080", dry_run=True)

In [ ]:
# Query 19: Service DNS and connectivity
kai("What is the internal DNS name for the nginx-demo-svc service in lab6-demo namespace and how would other pods reach it?")

In [ ]:
# Query 20: Service selector mismatch diagnostics
kai("Check if there are any services in lab6-demo that have no matching pod endpoints — this indicates a selector mismatch")

---

### 5.6 — Advanced Diagnostics: Cluster-Wide Health

These queries demonstrate kubectl-ai's power for holistic cluster analysis.

In [ ]:
# Query 21: Overall cluster health
kai("Give me an overall health summary of the cluster: nodes status, pod counts by state, and any warning events")

In [ ]:
# Query 22: Recent events and warnings
kai("Show me all Warning events from the last hour across all namespaces sorted by most recent")

In [ ]:
# Query 23: Resource pressure
kai("Which nodes have the highest pod count and are closest to their capacity?")

In [ ]:
# Query 24: Pending pods analysis
kai("Find any pods stuck in Pending state and explain the likely reason — check for resource constraints or node affinity issues")

In [ ]:
# Query 25: CrashLoopBackOff detection
kai("Find all pods with CrashLoopBackOff or ImagePullBackOff status and suggest what commands I should run to fix each one")

In [ ]:
# Query 26: RBAC and permissions
kai("List all service accounts in lab6-demo namespace and show if they have any role bindings")

In [ ]:
# Query 27: Complete namespace summary
kai("Give me a complete inventory of all resources in the lab6-demo namespace: pods, deployments, services, secrets, and configmaps")

---

### 5.7 — Using kubectl-ai for YAML Generation

One of kubectl-ai's most powerful features is generating Kubernetes YAML from natural language descriptions.

In [ ]:
# Query 28: Generate a deployment manifest
kai("""
Generate a Kubernetes deployment YAML for a Redis cache with:
- 1 replica
- image redis:7-alpine
- resource limits: 256Mi memory, 250m CPU
- namespace lab6-demo
""", dry_run=True)

In [ ]:
# Query 29: Generate a full stack (deployment + service + configmap)
kai("""
Generate a complete Kubernetes manifest for a simple web application with:
- A deployment with 2 replicas using httpd:alpine
- A ClusterIP service on port 80
- A configmap with APP_ENV=production and LOG_LEVEL=warn
- All in lab6-demo namespace
""", dry_run=True)

---

## 🧹 Part 6: Cleanup Lab Resources

In [ ]:
%%bash
# Clean up lab demo namespace
echo "🧹 Cleaning up lab6-demo resources..."
kubectl delete namespace lab6-demo --ignore-not-found=true
echo "✅ lab6-demo namespace deleted"

---

## 📝 Lab Summary

### What We Accomplished

| Task | Status |
|---|---|
| Installed Ollama locally | ✅ |
| Pulled llama3.2 model | ✅ |
| Installed kubectl-ai plugin | ✅ |
| Configured kubectl-ai with local Ollama | ✅ |
| Natural language Pod management | ✅ |
| Natural language Deployment management | ✅ |
| Natural language Secret/ConfigMap ops | ✅ |
| Natural language Service diagnostics | ✅ |
| Cluster-wide health queries | ✅ |
| YAML generation from natural language | ✅ |

### Key Takeaways

1. **Local LLMs** (via Ollama) enable AI-powered ops without sending cluster data to external services — critical for security-sensitive environments.

2. **kubectl-ai** bridges the gap between natural language intent and precise Kubernetes commands, reducing the cognitive load on operators.

3. **Troubleshooting with AI** is most effective when combined with structured cluster events, pod logs, and resource descriptions — kubectl-ai orchestrates all of these.

4. **Always use `--dry-run`** for potentially destructive operations until you're confident in the generated command.

---

## 🏋️ Practice Exercises

Try these additional queries on your own:

1. `"Create a namespace called 'production' with resource quotas limiting it to 4 CPUs and 8Gi memory"`
2. `"Find all deployments across all namespaces that have fewer ready replicas than desired"`
3. `"Generate a Kubernetes Job that runs a database migration script using image postgres:15"`
4. `"Show me all PersistentVolumeClaims in the cluster and their bound status"`
5. `"What are all the container images currently running in the cluster? Are any using the 'latest' tag?"`

---

## 📚 References

- [kubectl-ai GitHub Repository](https://github.com/GoogleCloudPlatform/kubectl-ai)
- [Ollama Documentation](https://ollama.com/docs)
- [Meta llama3.2 Model Card](https://ollama.com/library/llama3.2)
- [Kubernetes kubectl Reference](https://kubernetes.io/docs/reference/kubectl/)
- [AIOps Session 6 Lecture Slides](#) *(see course materials)*